# Restaurant AI - Analytics Engine

This notebook covers wait time estimation, staff detection, and generating actionable insights.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time
import json

from utils.detection import PersonDetector, get_video_info
from utils.tracking import DeepSORTTracker
from utils.analytics import (
    ZoneManager, FootfallCounter, StaffDetector,
    WaitTimeAnalyzer, StaffEfficiencyAnalyzer
)
from utils.visualization import AnalyticsEngine

## 2. Initialize All Components

In [ ]:
# Initialize detector and tracker
detector = PersonDetector(model_path='yolov8n.pt', confidence=0.5)
tracker = DeepSORTTracker(max_age=30, min_hits=3, iou_threshold=0.3)

# Initialize analytics components
zone_manager = ZoneManager()
footfall_counter = FootfallCounter(line_start=(320, 0), line_end=(320, 480))
staff_detector = StaffDetector(hsv_lower=(0, 0, 0), hsv_upper=(180, 50, 80))  # Dark clothing
wait_analyzer = WaitTimeAnalyzer()
staff_efficiency = StaffEfficiencyAnalyzer()
analytics_engine = AnalyticsEngine()

print("All components initialized")

## 3. Define Zones

In [ ]:
# Define zones (adjust coordinates based on your video resolution)
# Example for 640x480 video
zone_manager.add_zone('entrance', [
    (200, 400), (400, 400), (400, 450), (200, 450)
], (0, 255, 255))

zone_manager.add_zone('waiting', [
    (100, 250), (250, 250), (250, 350), (100, 350)
], (0, 255, 0))

zone_manager.add_zone('dining', [
    (400, 50), (620, 50), (620, 350), (400, 350)
], (255, 0, 0))

print("Zones defined:", list(zone_manager.zones.keys()))

## 4. Process Video and Collect Data

In [ ]:
VIDEO_PATH = '../data/sample_video.mp4'
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("Video not found. Using camera.")
    cap = cv2.VideoCapture(0)

video_info = get_video_info(cap)
print(f"Video info: {video_info}")

# Processing parameters
max_frames = 100  # Process up to 100 frames
frame_count = 0
prev_positions = {}
track_zones = {}
staff_track_ids = set()

start_time = time.time()

while frame_count < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    
    timestamp = time.time()
    
    # Detect persons
    boxes, confidences, class_ids = detector.detect(frame)
    
    # Detect staff
    staff_flags = staff_detector.detect_staff(frame, boxes)
    
    # Update tracker
    tracks, track_boxes, track_ids = tracker.update(
        boxes, confidences, frame, timestamp
    )
    
    # Process each track
    for i, track in enumerate(tracks):
        # Track zones
        zone = zone_manager.get_zone_at(track.center)
        if zone:
            zone_manager.update_track_zone(track.track_id, track.center, timestamp)
            
            if track.track_id not in track_zones:
                track_zones[track.track_id] = []
            if not track_zones[track.track_id] or track_zones[track.track_id][-1] != zone:
                track_zones[track.track_id].append(zone)
        
        # Staff detection
        if i < len(staff_flags) and staff_flags[i]:
            track.is_staff = True
            staff_track_ids.add(track.track_id)
            staff_efficiency.update_staff(
                track.track_id, track.center, timestamp, True
            )
        
        # Update wait times
        if zone == 'waiting' and track.track_id not in wait_analyzer.wait_start:
            wait_analyzer.start_wait(track.track_id, timestamp)
        elif zone == 'dining' and track.track_id in wait_analyzer.wait_start:
            wait_time = wait_analyzer.end_wait(track.track_id, timestamp)
            if wait_time:
                print(f"Track {track.track_id}: Wait time = {wait_time:.1f}s")
    
    # Update footfall counter
    for track in tracks:
        if track.track_id in prev_positions:
            result = footfall_counter.update(
                track.track_id, prev_positions[track.track_id], track.center
            )
        prev_positions[track.track_id] = track.center
    
    # Record footfall data periodically
    if frame_count % 10 == 0:
        analytics_engine.record_footfall(
            timestamp,
            footfall_counter.entries,
            footfall_counter.exits
        )
    
    frame_count += 1
    if frame_count % 20 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
processing_time = time.time() - start_time
print(f"\nProcessing complete: {frame_count} frames in {processing_time:.2f}s")

## 5. Generate Insights

In [ ]:
# Get current statistics
footfall_stats = footfall_counter.get_stats()
wait_stats = wait_analyzer.get_wait_time_distribution()
staff_metrics = staff_efficiency.get_efficiency_metrics()

current_customers = footfall_stats['net']
staff_count = len(staff_track_ids)
avg_wait_time = wait_stats.get('mean', 0)
avg_idle_time = staff_metrics.get('avg_idle_time', 0)

print("=" * 50)
print("CURRENT STATISTICS")
print("=" * 50)
print(f"Entries: {footfall_stats['entries']}")
print(f"Exits: {footfall_stats['exits']}")
print(f"Current customers: {current_customers}")
print(f"Staff count: {staff_count}")
print(f"Average wait time: {avg_wait_time:.1f}s ({avg_wait_time/60:.1f} min)")
print(f"Average idle time: {avg_idle_time:.1f}s")
print("=" * 50)

In [ ]:
# Generate insights
insights = analytics_engine.generate_insights(
    current_customers=current_customers,
    staff_count=staff_count,
    avg_wait_time=avg_wait_time,
    avg_idle_time=avg_idle_time
)

print("\n" + "=" * 50)
print("INSIGHTS AND RECOMMENDATIONS")
print("=" * 50)

if insights['warnings']:
    print("\nWarnings:")
    for warning in insights['warnings']:
        print(f"  - {warning}")

if insights['recommendations']:
    print("\nRecommendations:")
    for rec in insights['recommendations']:
        print(f"  - {rec}")

print("\nMetrics:")
for key, value in insights['metrics'].items():
    print(f"  {key}: {value}")

## 6. Peak Hours Analysis

In [ ]:
# Analyze peak hours
peak_analysis = analytics_engine.analyze_peak_hours()

if peak_analysis:
    print("Peak Hours Analysis:")
    print(f"  Peak hour: {peak_analysis.get('peak_hour', 'N/A')}")
    print(f"  Peak entries: {peak_analysis.get('peak_entries', 'N/A')}")
    
    hourly_dist = peak_analysis.get('hourly_distribution', {})
    if hourly_dist:
        print("\n  Hourly distribution:")
        for hour, count in sorted(hourly_dist.items()):
            print(f"    Hour {hour}: {count} entries")

## 7. Save Report

In [ ]:
# Save analytics report
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

analytics_engine.save_report(str(output_dir / 'analytics_report.json'))
print(f"Report saved to {output_dir / 'analytics_report.json'}")

# Also save footfall data
analytics_engine.export_to_csv(str(output_dir / 'footfall_data.csv'))
print(f"Footfall data saved to {output_dir / 'footfall_data.csv'}")

## Summary

This notebook covers:
- Complete video processing pipeline
- Wait time estimation (waiting zone to dining zone)
- Staff detection using color filtering
- Staff efficiency metrics (idle time, movement)
- Actionable insights generation
- Peak hours analysis
- Report export to JSON/CSV